<h1>Demand forecasting with BigQuery and TensorFlow</h1>

In this notebook, we will develop a machine learning model to predict the demand for taxi cabs in New York.

To develop the model, we will need to get historical data of taxicab usage. This data exists in BigQuery. Let's start by looking at the schema.

## Set up

In [ ]:
%pip install -q pandas-gbq tensorflow

**Restart** the kernel after installation.


In [ ]:
PROJECT = 'qwiklabs-gcp-04-15e0da53c189'   # CHANGE this to your GCP project
BUCKET  = PROJECT + '-ml'
REGION  = 'us-central1'            # CHANGE this to the region you want to use

In [ ]:
import os
os.environ['PROJECT'] = PROJECT
os.environ['BUCKET'] = BUCKET
os.environ['REGION'] = REGION

In [ ]:
%%bash
gcloud config set project ${PROJECT}
gcloud config set compute/region ${REGION}

In [ ]:
def query_to_dataframe(query):
  import pandas_gbq
  return pandas_gbq.read_gbq(query, dialect='standard', project_id=PROJECT)


## Explore table

In [ ]:
import pandas as pd
import numpy as np
import shutil

In [ ]:
query_to_dataframe("""
SELECT * FROM `bigquery-public-data.new_york.tlc_yellow_trips_2015` LIMIT 10
""")

<h2> Analyzing taxicab demand </h2>

Let's pull the number of trips for each day in the 2015 dataset using Standard SQL.

In [ ]:
query_to_dataframe("""
SELECT 
  EXTRACT (DAYOFYEAR from pickup_datetime) AS daynumber
FROM `bigquery-public-data.new_york.tlc_yellow_trips_2015` 
LIMIT 5
""")

<h3> Modular queries and Pandas dataframe </h3>

Let's use the total number of trips as our proxy for taxicab demand (other reasonable alternatives are total trip_distance or total fare_amount).  It is possible to predict multiple variables using Tensorflow, but for simplicity, we will stick to just predicting the number of trips.

We will give our query a name 'taxiquery' and have it use an input variable '$YEAR'. We can then invoke the 'taxiquery' by giving it a YEAR.  The to_dataframe() converts the BigQuery result into a <a href='http://pandas.pydata.org/'>Pandas</a> dataframe.

In [ ]:
def taxiquery(year):
  return """
WITH trips AS (
  SELECT EXTRACT (DAYOFYEAR from pickup_datetime) AS daynumber 
  FROM `bigquery-public-data.new_york.tlc_yellow_trips_*`
  where _TABLE_SUFFIX = '{}'
)
SELECT daynumber, COUNT(1) AS numtrips FROM trips
GROUP BY daynumber ORDER BY daynumber
""".format(year)

In [ ]:
trips = query_to_dataframe(taxiquery(2015))
trips[:5]

<h3> Benchmark </h3>

Often, a reasonable estimate of something is its historical average. We can therefore benchmark our machine learning model against the historical average.

In [ ]:
avg = np.mean(trips['numtrips'])
print('Just using average={0} has RMSE of {1}'.format(avg, np.sqrt(np.mean((trips['numtrips'] - avg)**2))))

The mean here is about 400,000 and the root-mean-square-error (RMSE) in this case is about 52,000. In other words, if we were to estimate that there are 400,000 taxi trips on any given day, that estimate is will be off on average by about 52,000 in either direction.
  
Let's see if we can do better than this -- our goal is to make predictions of taxicab demand whose RMSE is lower than 52,000.

What kinds of things affect people's use of taxicabs?

<h2> Weather data </h2>

We suspect that weather influences how often people use a taxi. Perhaps someone who'd normally walk to work would take a taxi if it is very cold or rainy.

One of the advantages of using a global data warehouse like BigQuery is that you get to mash up unrelated datasets quite easily.

In [ ]:
query_to_dataframe("""
SELECT * FROM `bigquery-public-data.noaa_gsod.stations`
WHERE state = 'NY' AND wban != '99999' AND name LIKE '%LA GUARDIA%'
""")

<h3> Variables </h3>

Let's pull out the minimum and maximum daily temperature (in Fahrenheit) as well as the amount of rain (in inches) for La Guardia airport.

In [ ]:
def wxquery(year):
  return """
SELECT EXTRACT (DAYOFYEAR FROM CAST(CONCAT('{0}','-',mo,'-',da) AS TIMESTAMP)) AS daynumber,
       MIN(EXTRACT (DAYOFWEEK FROM CAST(CONCAT('{0}','-',mo,'-',da) AS TIMESTAMP))) dayofweek,
       MIN(min) mintemp, MAX(max) maxtemp, MAX(IF(prcp=99.99,0,prcp)) rain
FROM `bigquery-public-data.noaa_gsod.gsod*`
WHERE stn='725030' AND _TABLE_SUFFIX = '{0}'
GROUP BY 1 ORDER BY daynumber DESC
""".format(year)

In [ ]:
weather = query_to_dataframe(wxquery(2015))
weather[:5]

<h3> Merge datasets </h3>

Let's use Pandas to merge (combine) the taxi cab and weather datasets day-by-day.

In [ ]:
data = pd.merge(weather, trips, on='daynumber')
data[:5]

<h3> Exploratory analysis </h3>

Is there a relationship between maximum temperature and the number of trips?

In [ ]:
j = data.plot(kind='scatter', x='maxtemp', y='numtrips')

The scatterplot above doesn't look very promising. There appears to be a weak downward trend, but it's also quite noisy.

Is there a relationship between the day of the week and the number of trips?

In [ ]:
j = data.plot(kind='scatter', x='dayofweek', y='numtrips')

Hurrah, we seem to have found a predictor. It appears that people use taxis more later in the week. Perhaps New Yorkers make weekly resolutions to walk more and then lose their determination later in the week, or maybe it reflects tourism dynamics in New York City.

Perhaps if we took out the <em>confounding</em> effect of the day of the week, maximum temperature will start to have an effect. Let's see if that's the case:

In [ ]:
j = data[data['dayofweek'] == 7].plot(kind='scatter', x='maxtemp', y='numtrips')

Removing the confounding factor does seem to reflect an underlying trend around temperature. But ... the data are a little sparse, don't you think?  This is something that you have to keep in mind -- the more predictors you start to consider (here we are using two: day of week and maximum temperature), the more rows you will need so as to avoid <em> overfitting </em> the model.

<h3> Adding 2014 and 2016 data </h3>

Let's add in 2014 and 2016 data to the Pandas dataframe.  Note how useful it was for us to modularize our queries around the YEAR.

In [ ]:
data2 = data # 2015 data
for year in [2014, 2016]:
    weather = query_to_dataframe(wxquery(year))
    trips = query_to_dataframe(taxiquery(year))
    data_for_year = pd.merge(weather, trips, on='daynumber')
    data2 = pd.concat([data2, data_for_year])
data2.describe()

In [ ]:
j = data2[data2['dayofweek'] == 7].plot(kind='scatter', x='maxtemp', y='numtrips')

The data do seem a bit more robust.  If we had even more data, it would be better of course. But in this case, we only have 2014-2016 data for taxi trips, so that's what we will go with.

<h2> Machine Learning with Tensorflow </h2>

We'll use 80% of our dataset for training and 20% of the data for testing the model we have trained. Let's shuffle the rows of the Pandas dataframe so that this division is random.  The predictor (or input) columns will be every column in the database other than the number-of-trips (which is our target, or what we want to predict).

The machine learning models that we will use -- linear regression and neural networks -- both require that the input variables are numeric in nature.

The day of the week, however, is a categorical variable (i.e. Tuesday is not really greater than Monday). So, we should create separate columns for whether it is a Monday (with values 0 or 1), Tuesday, etc.

Against that, we do have limited data (remember: the more columns you use as input features, the more rows you need to have in your training dataset), and it appears that there is a clear linear trend by day of the week. So, we will opt for simplicity here and use the data as-is.  Try uncommenting the code that creates separate columns for the days of the week and re-run the notebook if you are curious about the impact of this simplification.

In [ ]:
import tensorflow as tf
shuffled = data2.sample(frac=1, random_state=13)
# It would be a good idea, if we had more data, to treat the days as categorical variables
# with the small amount of data, we have though, the model tends to overfit
#predictors = shuffled.iloc[:,2:5]
#for day in range(1,8):
#  matching = shuffled['dayofweek'] == day
#  key = 'day_' + str(day)
#  predictors[key] = pd.Series(matching, index=predictors.index, dtype=float)
predictors = shuffled.iloc[:,1:5]
predictors[:5]

In [ ]:
shuffled[:5]

In [ ]:
targets = shuffled.iloc[:,5]
targets[:5]

Let's update our benchmark based on the 80-20 split and the larger dataset.

In [ ]:
trainsize = int(len(shuffled['numtrips']) * 0.8)
avg = np.mean(shuffled['numtrips'][:trainsize])
rmse = np.sqrt(np.mean((targets[trainsize:] - avg)**2))
print('Just using average={0} has RMSE of {1}'.format(avg, rmse))

<h2> Linear regression with tf.contrib.learn </h2>

We scale the number of taxicab rides by 400,000 so that the model can keep its predicted values in the [0-1] range. The optimization goes a lot faster when the weights are small numbers.  We save the weights into ./trained_model_linear and display the root mean square error on the test dataset.

In [ ]:
SCALE_NUM_TRIPS = 600000.0
trainsize = int(len(shuffled['numtrips']) * 0.8)
npredictors = len(predictors.columns)
noutputs = 1

X_train = predictors[:trainsize].to_numpy(dtype=np.float32)
y_train = targets[:trainsize].to_numpy(dtype=np.float32) / SCALE_NUM_TRIPS
X_test = predictors[trainsize:].to_numpy(dtype=np.float32)
y_test = targets[trainsize:].to_numpy(dtype=np.float32)

norm = tf.keras.layers.Normalization()
norm.adapt(X_train)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(npredictors,)),
    norm,
    tf.keras.layers.Dense(noutputs)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.05), loss='mse')

print("starting to train ...")
model.fit(X_train, y_train, epochs=200, batch_size=32, verbose=0)
model.save('./trained_model_linear.keras')

pred = model.predict(X_test, verbose=0).flatten() * SCALE_NUM_TRIPS
rmse = np.sqrt(np.mean(np.power((y_test - pred), 2)))
print('LinearRegression has RMSE of {0}'.format(rmse))


The RMSE here (57K) is lower than the benchmark (62K) indicates that we are doing about 10% better with the machine learning model than we would be if we were to just use the historical average (our benchmark).

<h2> Neural network with tf.contrib.learn </h2>

Let's make a more complex model with a few hidden nodes.

In [ ]:
SCALE_NUM_TRIPS = 600000.0
trainsize = int(len(shuffled['numtrips']) * 0.8)
npredictors = len(predictors.columns)
noutputs = 1

X_train = predictors[:trainsize].to_numpy(dtype=np.float32)
y_train = targets[:trainsize].to_numpy(dtype=np.float32) / SCALE_NUM_TRIPS
X_test = predictors[trainsize:].to_numpy(dtype=np.float32)
y_test = targets[trainsize:].to_numpy(dtype=np.float32)

norm = tf.keras.layers.Normalization()
norm.adapt(X_train)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(npredictors,)),
    norm,
    tf.keras.layers.Dense(5, activation='relu'),
    tf.keras.layers.Dense(5, activation='relu'),
    tf.keras.layers.Dense(noutputs)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse')

print("starting to train ...")
model.fit(X_train, y_train, epochs=200, batch_size=32, verbose=0)
model.save('./trained_model.keras')

pred = model.predict(X_test, verbose=0).flatten() * SCALE_NUM_TRIPS
rmse = np.sqrt(np.mean((y_test - pred) ** 2))
print('Neural Network Regression has RMSE of {0}'.format(rmse))


Using a neural network results in similar performance to the linear model when I ran it -- it might be because there isn't enough data for the NN to do much better. (NN training is a non-convex optimization, and you will get different results each time you run the above code).

<h2> Running a trained model </h2>

So, we have trained a model, and saved it to a file. Let's use this model to predict taxicab demand given the expected weather for three days.

Here we make a Dataframe out of those inputs, load up the saved model (note that we have to know the model equation -- it's not saved in the model file) and use it to predict the taxicab demand.

In [ ]:
SCALE_NUM_TRIPS = 600000.0

input_df = pd.DataFrame.from_dict(data={
    'dayofweek': [4, 5, 6],
    'mintemp': [60, 40, 50],
    'maxtemp': [70, 90, 60],
    'rain': [0, 0.5, 0]
})

model = tf.keras.models.load_model('./trained_model_linear.keras')
pred = model.predict(input_df.to_numpy(dtype=np.float32), verbose=0).flatten() * SCALE_NUM_TRIPS
print(pred)


Looks like we should tell some of our taxi drivers to take the day off on Thursday (day=5). No wonder -- the forecast calls for extreme weather fluctuations on Thursday.

Copyright 2017 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License